In [ ]:
import os
from dotenv import load_dotenv
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.functions import year, col
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DateType
from functools import reduce

os.environ["JAVA_HOME"] = "/Library/Java/JavaVirtualMachines/zulu-17.jdk/Contents/Home" # ignore this using your java setting




<h1>
DATA OVERIVIEW
</h1>

<text>The Following dataset is a subset data from "NOAA Global Surface Summary of Day". It covers year from 2022-2025, 4 years in total, and includes data for 50 unique weather station. 

The following are data elements included in the dataset (as available from each station), and how they are denoted in dataframe

* Mean temperature (.1 Fahrenheit). TEMP
* Mean dew point (.1 Fahrenheit). DEWP
* Mean sea level pressure (.1 mb). SLP
* Mean station pressure (.1 mb). STP
* Mean visibility (.1 miles). VISIB
* Mean wind speed (.1 knots), WDSP
* Maximum sustained wind speed (.1 knots). MXSPD
* Maximum wind gust (.1 knots). GUST
* Maximum temperature (.1 Fahrenheit). MAX
* Minimum temperature (.1 Fahrenheit). MIN
* Precipitation amount (.01 inches). PRCP
* Snow depth (.1 inches). SNDP
* Indicator for occurrence of: Fog. Rain or Drizzle. Snow or Ice Pellets. Hail. Thunder. Tornado/Funnel Cloud. FRSHTT
</text>

<h1>
I) DATA EXTRACTION
<h1>

In [ ]:
""" 
The following code will read the data, year 2022-2025, from azure blob and turn them into spark Dataframe

- Running this code will automatically overwrite the data into raw data
"""


spark = SparkSession.builder.appName("NOAA Pipeline").getOrCreate()
storage_account_name = "noaapipiline"
container_name = "rawdata"

# replace sas token, expired on April 20th
load_dotenv()
sas_token = os.environ.get("SAS_TOKEN") 
if not sas_token:
    raise ValueError("No valid SAS token")

spark.conf.set(
    f"fs.azure.sas.{container_name}.{storage_account_name}.blob.core.windows.net", sas_token
)

# data schema define 
myschema = StructType([
    StructField("STATION", IntegerType(), False),
    StructField("DATE", DateType(), True),
    StructField("LATITUDE", DoubleType(), True),
    StructField("LONGITUDE", DoubleType(), True),
    StructField("ELEVATION", DoubleType(), True),
    StructField("NAME", StringType(), True),
    StructField("TEMP", DoubleType(), True),
    StructField("TEMP_ATTRIBUTES", IntegerType(), True),
    StructField("DEWP", DoubleType(), True),
    StructField("DEWP_ATTRIBUTES", IntegerType(), True),
    StructField("SLP", DoubleType(), True),
    StructField("SLP_ATTRIBUTES", IntegerType(), True),
    StructField("STP", DoubleType(), True),
    StructField("STP_ATTRIBUTES", IntegerType(), True),
    StructField("VISIB", DoubleType(), True),
    StructField("VISIB_ATTRIBUTES", IntegerType(), True),
    StructField("WDSP", DoubleType(), True),
    StructField("WDSP_ATTRIBUTES", IntegerType(), True),
    StructField("MXSPD", DoubleType(), True),
    StructField("GUST", DoubleType(), True),
    StructField("MAX", DoubleType(), True),
    StructField("MAX_ATTRIBUTES", StringType(), True),
    StructField("MIN", DoubleType(), True),
    StructField("MIN_ATTRIBUTES", StringType(), True),
    StructField("PRCP", DoubleType(), True),
    StructField("PRCP_ATTRIBUTES", StringType(), True),
    StructField("SNDP", DoubleType(), True),
    StructField("FRSHTT", StringType(), True),
])

# read file from azureblob and return as spark dataframe
def read_csv(year:int) -> DataFrame:
    print(f"Extracting files from azure blob for {year}")
    file_path = f"wasbs://{container_name}@{storage_account_name}.blob.core.windows.net/{year}"

    df = (spark.read
      .format("csv")
      .option("header", "true")
      .schema(myschema)
      .load(file_path))

    print("Successful extracting the file")
    return df

df_2022 = read_csv(2022)
df_2023 = read_csv(2023)
df_2024 = read_csv(2024)
df_2025 = read_csv(2025)

print("=" * 20)
print("Finishing extracting all files for all years")
print("=" * 20)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/07 17:22:08 WARN Utils: Your hostname, MacBook-Air-cua-Tran.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.101 instead (on interface en0)
26/04/07 17:22:08 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/07 17:22:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Extracting files from azure blob for 2022
Successful extracting the file
Extracting files from azure blob for 2023
Successful extracting the file
Extracting files from azure blob for 2024
Successful extracting the file
Extracting files from azure blob for 2025
Successful extracting the file
Finishing extracting all files for all years


<h1>
II) DATA_EXPLORATION
<h1>

In [3]:
# combine all year dataframe into one
df_list = [df_2022,df_2023, df_2024, df_2025]
df_combine = reduce(DataFrame.union, df_list)

In [4]:
"""
The following code is a data exploration for year from 2022-2025
By running the code, it will process the following task

- coutning total number of records
- counting numbers of unique stations
- counting number of years that data covers
- finding all record that have null values in TEMP columns, mean temperature
- finding all records that have mean temperature bigger than max temperature, TEMP > MAX
- finding all records that have mean temperature smaller than min temperature, TEMP < MIN
"""



# counting total records
number_record = df_combine.count()

print("=" * 20)
print(f"number of records")
print(number_record)
print("=" * 20)

# counting unique stations
stations = df_combine.select("STATION").distinct().count()

print("=" * 20)
print(f"number of unique weather station")
print(stations)
print("=" * 20)

# findings length of year covers
df_year = df_combine.withColumn("YEAR", year("DATE"))
years_cover = df_year.select("YEAR").distinct().count()

print("=" * 20)
print(f"number of year cover")
print(years_cover)
print("=" * 20)

# check null and invalid values for temperature field
df_check = (
    df_combine.filter((col("TEMP").isNull()) | (col("TEMP") < col("MIN")) | (col("TEMP") > col("MAX")))
)
invalid_temp_records = df_check.count()

print("="*20)
print(f"sample of invalid data in temperature field")
df_check.select("TEMP", "MAX", "MIN").show(10)
print(f"Number of records have null and invalid temperature:")
print(invalid_temp_records)
print("="*20)


number of records
48404


number of unique weather station
50


number of year cover
4


sample of invalid data in temperature field


+----+------+------+
|TEMP|   MAX|   MIN|
+----+------+------+
|46.4|9999.9|9999.9|
|41.0|9999.9|9999.9|
|44.6|9999.9|9999.9|
|46.4|9999.9|9999.9|
|48.2|9999.9|9999.9|
|48.2|9999.9|9999.9|
|50.0|9999.9|9999.9|
|53.6|9999.9|9999.9|
|50.0|9999.9|9999.9|
|48.2|9999.9|9999.9|
+----+------+------+
only showing top 10 rows
Number of records have null and invalid temperature:
153


In [ ]:
"""
This code will stores all the csv file into shared folder

Running this code will overwrite files in shared folder
"""

current_path = os.getcwd()
root_path = current_path.removesuffix("/task1")
stored_path = os.path.join(root_path, "shared")


df_combine.write.csv(path= stored_path, mode= "overwrite", header = True)
print("="*20)
print(f"Successful store data")
print("="*20)


Successful store data
